Termolab python


1.) Beregn massestrømmen av kuldemedia $\dot{m}, [\frac{kg}{s}]$, for hver målingsserie dere har registrert ved å bruke energibalansen til et åpent system:

$$
\dot{W}_{komp} = \dot{m}(h_{inn}-h_{ut})
$$
 
som gir oss
$$
\frac{\dot{W}_{komp}}{(h_{inn}-h_{ut})} = \dot{m}
$$
hvor 
$$
h_{ut} = h_{inn} + \frac{h_{2,is}-h_{inn}}{\eta_{is}}
$$
og 
$$
\eta_{is} = 0.0017\pi^3 - 0.0286\pi^2 + 0.1533\pi + 0.1642
$$
$$
\pi = \frac{p_{kondenser}}{p_{fordamper}}
$$

In [16]:
import pandas as pd
import numpy as np

# m trenger: W, h_inn, h_ut, 
# h_ut trenger: h_inn, h_2,is, eta_is
# eta_is trenger: p_kond, p_ford
# p_kond, p_ford trenger: t_kond, t_ford
# h_2,is trenger: 

df = pd.read_excel("merged log termo.xlsx", sheet_name="log")  

df.columns = ["time", "T1", "T2", "T3", "T4", "Thot", "Tcold", "W_komp", "Tkond", "Tford"]   
            #hh:mm:ss, degC,, degC, degC, degC, degC, degC,      W,       degC,    degC 
# finner W
W_komp = df["W_komp"].to_numpy()


In [ ]:
from CoolProp.CoolProp import PropsSI
#finner entalpier


fluid = "R134a"
kelvin = 273.15
manuel_tid= ["12:24:01", "12:25:59", "12:28:08", "12:30:06", "12:32:04"]

log_date = df["time"].dt.date.iloc[0]

start_time, end_time = pd.to_datetime([
    f"{log_date} {manuel_tid[0]}",
    f"{log_date} {manuel_tid[-1]}"])

manuel_tid = df.loc[df["time"].between(start_time, end_time)].copy()

def entalpi(data, Ta, Tb):
    
    """
    input: data -> dataset
    Ta er første temperatur, Tb er andre temperatur
    
    1. lager kopi av dataset
    2. finner fordampnings- og konsenseringstrykk
    3. bruker trykkene for å finne den isentropiske virkningsgraden
    4. finenr h1, h2, h2is ved hjelp av dataene i forrige steg
    5. returnerer dataset med original data + nye entalpier
    """

    data = data.copy()
    T_suc  = data[Ta].to_numpy() + kelvin   # [K]
    p_ford = PropsSI("P", "T", data["Tford"].to_numpy() + kelvin, "Q", 1, fluid)   # [Pa]
    p_kond = PropsSI("P", "T", data["Tkond"].to_numpy() + kelvin, "Q", 1, fluid)   # [Pa]

    data["superheat_K"] = data[Ta] - data[Tb]            
    data["pi"] = p_kond / p_ford
    pi = data["pi"]
    data["eta_is"] = 0.0017*pi**3 - 0.0286*pi**2 + 0.1533*pi + 0.1642

    h1   = PropsSI("H", "T", T_suc, "P", p_ford, fluid)       # [J/kg]
    s1   = PropsSI("S", "T", T_suc, "P", p_ford, fluid)       # [J/kgK]
    h2is = PropsSI("H", "P", p_kond, "S", s1, fluid)          # isentropisk 
    h2   = h1 + (h2is - h1) / data["eta_is"].to_numpy()       # ekte

    data["h1"], data["h2is"], data["h2"] = h1, h2is, h2       # [J/kg]
    return data

res = entalpi(manuel_tid, "T1", "Tkond")
res_manual = res[res["time"].dt.strftime("%H:%M:%S").isin(manuel_tid)]
print(res_manual[["time", "eta_is", "h1", "h2", "h2is"]])

                  time    eta_is             h1             h2           h2is
27 2026-10-06 12:24:01  0.396674  423448.046318  477115.550893  444736.562076
38 2026-10-06 12:25:59  0.411898  427709.421687  489917.418805  453332.749961
50 2026-10-06 12:28:08  0.421804  432295.215675  502613.469525  461955.740381
61 2026-10-06 12:30:06  0.428598  435674.298241  515587.507109  469924.934773
72 2026-10-06 12:32:04  0.429792  438680.891418  524540.959219  475582.827625


In [22]:
res["m_dot"] = res["W_komp"] / (res["h2"] - res["h1"])     # [kg/s]
res["m_dot_e3"] = res["m_dot"] * 1e3                        # [1e-3 kg/s]


res_manual = res[res["time"].isin(manuel_tid)]

print(res_manual["m_dot"])

27    0.002739
38    0.002395
50    0.002147
61    0.001915
72    0.001817
Name: m_dot, dtype: float64


2. Beregne varmeytelse av kondensator i $Q_{kond}$, [W], for hver målingsserie dere har tatt ved å
bruke energibalansen til et åpent system:


$$
\dot{Q}_{kond} = \dot{m}(h_{ut}-h_{inn})
$$

$
h_{inn} – \text{ entalpien inn på kondensator}, [\frac{J}{kg}]$
, bruk T2 temperaturen og kondenseringstrykk $p_{kond}$.

$h_{ut} - \text{ entalpien ut av kondensator} , [\frac{J}{kg}]$
,bruk kondenseringstrykk, $p_{kond}$, og T3 temperaturen.

For å kompensere for varmetap mellom rør og omgivelsestemperaturen til kondensatoren, så
anbefales det at dere legger til 4.8 K til temperaturverdiene I T3 og T2.
Altså at dere tar:
Temperatur T3 = T3 + 4.8 K (VIKTIG: koeffisienten brukes ikke hvis T3 + 4.8 K> metnings
temperatur).
Temperatur T2 = T2 + 4.8 K

In [23]:
# 5.1.2: Kondensatorytelse for de valgte maletidspunktene
# Endre tidene eller korreksjonen nedenfor ved behov.
from CoolProp.CoolProp import PropsSI

maletider_kond = ["12:24:01", "12:25:59", "12:28:08", "12:30:06", "12:32:04"]
korreksjon_K = 4.8
resultater_kond = []

for tid in maletider_kond:
    malinger = df.loc[df["time"].dt.strftime("%H:%M:%S") == tid]
    if len(malinger) != 1:
        raise ValueError(f"Forventet en maling kl. {tid}, fant {len(malinger)}")
    mal = malinger.iloc[0]

    # Mettetrykk fra kondenserings- og fordampingstemperaturene.
    p_kond = PropsSI("P", "T", mal["Tkond"] + kelvin, "Q", 0, fluid)
    p_ford = PropsSI("P", "T", mal["Tford"] + kelvin, "Q", 1, fluid)

    # Massestrom fra kompressorens energibalanse.
    h1 = PropsSI("H", "T", mal["T1"] + kelvin, "P", p_ford, fluid)
    s1 = PropsSI("S", "T", mal["T1"] + kelvin, "P", p_ford, fluid)
    h2_is = PropsSI("H", "P", p_kond, "S", s1, fluid)
    pi = p_kond / p_ford
    eta_is = 0.0017*pi**3 - 0.0286*pi**2 + 0.1533*pi + 0.1642
    if eta_is <= 0:
        raise ValueError(f"Ugyldig isentropisk virkningsgrad ved {tid}: {eta_is}")
    h2_komp = h1 + (h2_is - h1) / eta_is
    m_dot = mal["W_komp"] / (h2_komp - h1)

    # Ut av kondensatoren: bruk +4.8 K bare hvis temperaturen ikke overstiger metning.
    T3_korr = mal["T3"] + korreksjon_K
    bruk_T3_korreksjon = T3_korr <= mal["Tkond"]
    T3_brukt = T3_korr if bruk_T3_korreksjon else mal["T3"]
    h3 = PropsSI("H", "T", T3_brukt + kelvin, "P", p_kond, fluid)

    # Estimatet brukt tidligere: kompressormodellens entalpi er innlop til kondensatoren.
    Q_kond_kompressor = m_dot * (h2_komp - h3)

    # Oppgaveteksten ber ogsa om T2 + 4.8 K som kondensatorinnlop.
    T2_korr = mal["T2"] + korreksjon_K
    h2_T2 = PropsSI("H", "T", T2_korr + kelvin, "P", p_kond, fluid)
    Q_kond_T2 = m_dot * (h2_T2 - h3)
    sjekk_T2_T3 = "SJEKK T2/T3: innlop <= utlop" if T2_korr <= T3_brukt else "OK"

    resultater_kond.append({
        "Indeks": mal.name,
        "time": mal["time"],
        "T3 + 4.8 [C]": T3_korr,
        "T3 brukt [C]": T3_brukt,
        "T_kond [C]": mal["Tkond"],
        "+4.8 K pa T3": bruk_T3_korreksjon,
        "m_dot [kg/s]": m_dot,
        "Q_kond kompressormodell [W]": Q_kond_kompressor,
        "Q_kond fra T2 [W]": Q_kond_T2,
        "Kontroll T2/T3": sjekk_T2_T3,
    })

resultater_kond = pd.DataFrame(resultater_kond).set_index("Indeks")
resultater_kond.index.name = None
print(resultater_kond)

[{'Tid': '12:24:01', 'T3 + 4.8 [C]': np.float64(38.55), 'T3 brukt [C]': np.float64(38.55), 'T_kond [C]': np.float64(45.0), '+4.8 K pa T3': np.True_, 'm_dot [kg/s]': np.float64(0.0027390876688871088), 'Q_kond kompressormodell [W]': np.float64(610.5293988358461), 'Q_kond fra T2 [W]': np.float64(-61.39859852193667), 'Kontroll T2/T3': 'SJEKK T2/T3: innlop <= utlop'}, {'Tid': '12:25:59', 'T3 + 4.8 [C]': np.float64(43.3), 'T3 brukt [C]': np.float64(43.3), 'T_kond [C]': np.float64(50.0), '+4.8 K pa T3': np.True_, 'm_dot [kg/s]': np.float64(0.002395190440185126), 'Q_kond kompressormodell [W]': np.float64(547.5505631412519), 'Q_kond fra T2 [W]': np.float64(-74.86554941320608), 'Kontroll T2/T3': 'SJEKK T2/T3: innlop <= utlop'}, {'Tid': '12:28:08', 'T3 + 4.8 [C]': np.float64(48.3), 'T3 brukt [C]': np.float64(48.3), 'T_kond [C]': np.float64(53.5), '+4.8 K pa T3': np.True_, 'm_dot [kg/s]': np.float64(0.0021473798300371155), 'Q_kond kompressormodell [W]': np.float64(501.80964347208277), 'Q_kond fra 